In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 학습 모델 저장을 위한 라이브러리
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = '/content/drive/MyDrive/은서님 파일/best_model_cd_2.dat'

In [4]:
with open(best_model_path, 'rb') as fp :
    best_model = pickle.load(fp)
    scalerX = pickle.load(fp)
    scaler_columns = pickle.load(fp)
    Encoder3 = pickle.load(fp)
    le = pickle.load(fp)

display(best_model)
display(scalerX)
display(scaler_columns)
display(le)
display(Encoder3)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

StandardScaler()

['_1순위카드이용금액',
 '연속유실적개월수_기본_24M_카드',
 '이용금액_R3M_신용체크',
 '이용금액_일시불_R12M',
 '정상청구원금_B0M',
 '정상청구원금_B2M',
 '정상청구원금_B5M',
 '청구금액_B0',
 '청구금액_R3M',
 '청구금액_R6M',
 '방문횟수_앱_R6M']

LabelEncoder()

LabelEncoder()

### Target

In [5]:
tg_df = pd.read_csv('/content/drive/MyDrive/Segment.csv')
tg_df = tg_df['Segment']
tg_df

,Segment
0,D
1,E
2,C
3,D
4,E
...,...
2399995,E
2399996,D
2399997,C
2399998,E


In [6]:
test_cols =['기준년월','ID','_1순위카드이용금액',
 '연속유실적개월수_기본_24M_카드',
 '이용금액_R3M_신용체크',
 '이용금액_일시불_R12M',
 '정상청구원금_B0M',
 '정상청구원금_B2M',
 '정상청구원금_B5M',
 '청구금액_B0',
 '청구금액_R3M',
 '청구금액_R6M',
 '방문횟수_앱_R6M']

In [8]:
# 예측할 데이터를 읽어온다.
df1 = pd.read_parquet('/content/drive/MyDrive/abcde/ab_cde_test.parquet')
df1 = df1[test_cols]
df1

,기준년월,ID,_1순위카드이용금액,연속유실적개월수_기본_24M_카드,이용금액_R3M_신용체크,이용금액_일시불_R12M,정상청구원금_B0M,정상청구원금_B2M,정상청구원금_B5M,청구금액_B0,청구금액_R3M,청구금액_R6M,방문횟수_앱_R6M
0,201807,TEST_00000,13852,5,21458,49063,6495,2012,3919,4931,11441,22151,1회 이상
1,201807,TEST_00001,11065,8,18681,7771,9644,5273,5723,10152,20522,32878,1회 이상
2,201807,TEST_00002,27071,24,40758,73003,11519,11366,11267,13223,50508,71867,1회 이상
3,201807,TEST_00003,4827,5,5255,7421,1375,1072,0,2112,4604,4986,1회 이상
4,201807,TEST_00004,8011,20,16148,10493,3951,1618,1347,4406,6788,10758,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,201812,TEST_99995,0,0,0,0,0,0,0,0,0,0,1회 이상
599996,201812,TEST_99996,1231,8,3110,2631,205,3736,992,359,1256,2237,10회 이상
599997,201812,TEST_99997,0,0,0,0,0,186,0,0,0,0,1회 이상
599998,201812,TEST_99998,63592,24,173263,423882,26308,23261,27335,21273,48141,108420,40회 이상


In [9]:
notE = pd.read_csv('/content/drive/MyDrive/은서님 파일/model3_notE.csv')
notE

,ID,Segment
0,TEST_00002,Not_E
1,TEST_00010,Not_E
2,TEST_00012,Not_E
3,TEST_00016,Not_E
4,TEST_00032,Not_E
...,...,...
16078,TEST_99957,Not_E
16079,TEST_99961,Not_E
16080,TEST_99982,Not_E
16081,TEST_99994,Not_E


In [10]:
# 1. 'Not_E'인 ID만 필터링
filtered_ids1 = notE[notE['Segment'] == 'Not_E']['ID']

# 2. df2에서 해당 ID만 골라오기
df2 = df1[df1['ID'].isin(filtered_ids1)]

In [11]:
df2

,기준년월,ID,_1순위카드이용금액,연속유실적개월수_기본_24M_카드,이용금액_R3M_신용체크,이용금액_일시불_R12M,정상청구원금_B0M,정상청구원금_B2M,정상청구원금_B5M,청구금액_B0,청구금액_R3M,청구금액_R6M,방문횟수_앱_R6M
2,201807,TEST_00002,27071,24,40758,73003,11519,11366,11267,13223,50508,71867,1회 이상
10,201807,TEST_00010,7544,1,45764,29326,1614,2714,3028,0,6847,12797,1회 이상
12,201807,TEST_00012,24425,24,37849,86805,6621,8468,11724,7283,33803,63042,1회 이상
16,201807,TEST_00016,42636,24,51440,207417,16155,16063,15913,12194,41328,77903,1회 이상
32,201807,TEST_00032,3885,24,54632,21910,633,916,1213,769,4259,11525,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599957,201812,TEST_99957,38644,21,61148,136892,14116,9573,15535,14349,28638,66775,40회 이상
599961,201812,TEST_99961,25961,24,30318,117578,10170,13173,14778,7195,26095,58282,1회 이상
599982,201812,TEST_99982,70955,24,94440,447150,26321,26305,24445,11710,65023,129230,1회 이상
599994,201812,TEST_99994,29020,8,31697,33505,18916,20198,19843,28423,98428,150417,1회 이상


In [12]:
notAB = pd.read_csv('/content/drive/MyDrive/은서님 파일/model3_notAB.csv')
notAB

,ID,Segment
0,TEST_00002,notAB
1,TEST_00010,notAB
2,TEST_00012,notAB
3,TEST_00016,notAB
4,TEST_00032,notAB
...,...,...
16063,TEST_99957,notAB
16064,TEST_99961,notAB
16065,TEST_99982,notAB
16066,TEST_99994,notAB


In [13]:
# 1. 'Not_E'인 ID만 필터링
filtered_ids2 = notAB[notAB['Segment'] == 'notAB']['ID']

# 2. df2에서 해당 ID만 골라오기
Xtest_df = df2[df2['ID'].isin(filtered_ids2)]

In [14]:
Xtest_df

,기준년월,ID,_1순위카드이용금액,연속유실적개월수_기본_24M_카드,이용금액_R3M_신용체크,이용금액_일시불_R12M,정상청구원금_B0M,정상청구원금_B2M,정상청구원금_B5M,청구금액_B0,청구금액_R3M,청구금액_R6M,방문횟수_앱_R6M
2,201807,TEST_00002,27071,24,40758,73003,11519,11366,11267,13223,50508,71867,1회 이상
10,201807,TEST_00010,7544,1,45764,29326,1614,2714,3028,0,6847,12797,1회 이상
12,201807,TEST_00012,24425,24,37849,86805,6621,8468,11724,7283,33803,63042,1회 이상
16,201807,TEST_00016,42636,24,51440,207417,16155,16063,15913,12194,41328,77903,1회 이상
32,201807,TEST_00032,3885,24,54632,21910,633,916,1213,769,4259,11525,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599957,201812,TEST_99957,38644,21,61148,136892,14116,9573,15535,14349,28638,66775,40회 이상
599961,201812,TEST_99961,25961,24,30318,117578,10170,13173,14778,7195,26095,58282,1회 이상
599982,201812,TEST_99982,70955,24,94440,447150,26321,26305,24445,11710,65023,129230,1회 이상
599994,201812,TEST_99994,29020,8,31697,33505,18916,20198,19843,28423,98428,150417,1회 이상


In [15]:
X_test = Xtest_df.drop(columns=['기준년월','ID'])
X_test = X_test[scaler_columns]

In [16]:
X_test

,_1순위카드이용금액,연속유실적개월수_기본_24M_카드,이용금액_R3M_신용체크,이용금액_일시불_R12M,정상청구원금_B0M,정상청구원금_B2M,정상청구원금_B5M,청구금액_B0,청구금액_R3M,청구금액_R6M,방문횟수_앱_R6M
2,27071,24,40758,73003,11519,11366,11267,13223,50508,71867,1회 이상
10,7544,1,45764,29326,1614,2714,3028,0,6847,12797,1회 이상
12,24425,24,37849,86805,6621,8468,11724,7283,33803,63042,1회 이상
16,42636,24,51440,207417,16155,16063,15913,12194,41328,77903,1회 이상
32,3885,24,54632,21910,633,916,1213,769,4259,11525,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...
599957,38644,21,61148,136892,14116,9573,15535,14349,28638,66775,40회 이상
599961,25961,24,30318,117578,10170,13173,14778,7195,26095,58282,1회 이상
599982,70955,24,94440,447150,26321,26305,24445,11710,65023,129230,1회 이상
599994,29020,8,31697,33505,18916,20198,19843,28423,98428,150417,1회 이상


In [17]:
X_test['방문횟수_앱_R6M'] = Encoder3.transform(X_test['방문횟수_앱_R6M'])

In [18]:
# 데이터 표준화
scaled_data = scalerX.transform(X_test)
scaled_data

array([[ 0.51822497,  0.87354078,  0.38005302, ...,  0.74991564,
         0.23900601, -0.2758293 ],
       [-0.55965122, -1.51938854,  0.54351191, ..., -0.62807283,
        -0.66259372, -0.2758293 ],
       [ 0.3721677 ,  0.87354078,  0.28506662, ...,  0.22268774,
         0.1043079 , -0.2758293 ],
       ...,
       [ 2.94058985,  0.87354078,  2.13290962, ...,  1.20802477,
         1.11455139, -0.2758293 ],
       [ 0.62580836, -0.7911057 ,  0.08418785, ...,  2.26232276,
         1.4379337 , -0.2758293 ],
       [ 2.53415761,  0.87354078,  4.70668512, ...,  0.67521055,
         0.79692331,  2.45983321]])

In [19]:
# 입력데이터를 test_X 변수에 담아준다.
X_text2 = scaled_data

In [20]:
# 예측한다.
y_pred = best_model.predict(X_text2)
y_pred_labels = le.inverse_transform(y_pred)
y_pred_labels

array(['D', 'D', 'D', ..., 'C', 'D', 'C'], dtype=object)

In [21]:
# 결과를 붙힌다음 저장한다.
# 예측할 데이터를 다시 불러온다.
result_df = Xtest_df
result_df['Segment'] = y_pred_labels

In [22]:
result_df2 = result_df[['ID', 'Segment']]

In [23]:
result_df2['Segment'].value_counts()

,count
Segment,
D,72852
C,23556


In [24]:
most_common_by_id = result_df2.groupby('ID')['Segment'] \
    .agg(lambda x: x.mode().iloc[0]) \
    .reset_index()

# 컬럼명 확인
most_common_by_id.columns = ['ID', 'Segment']

In [25]:
most_common_by_id

,ID,Segment
0,TEST_00002,D
1,TEST_00010,D
2,TEST_00012,D
3,TEST_00016,D
4,TEST_00032,D
...,...,...
16063,TEST_99957,D
16064,TEST_99961,D
16065,TEST_99982,C
16066,TEST_99994,D


In [26]:
most_common_by_id['Segment'].value_counts()

,count
Segment,
D,12040
C,4028


In [27]:
most_common_by_id.to_csv('/content/drive/MyDrive/은서님 파일/model8201_C,D.csv', index=False, encoding='utf-8-sig')
print('저장완료')

저장완료
